In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# 1. SymPy Settings and Symbol Definitions
# ==============================================================================

sp.init_printing(use_unicode=True)

# Ορίζουμε το z_inv ως z^{-1} για να δουλεύουμε απευθείας με τις δυνάμεις του z^-1
z_inv = sp.Symbol('z^{-1}', complex=True)
n = sp.Symbol('n', integer=True, nonnegative=True)

print("=== METHOD: LTI System Response via Z-Transform (Exact Symbolic Derivation) ===")

# 1. Impulse Response h[n] και Συνάρτηση Μεταφοράς H(z^-1) από τον ορισμό
h_n = (sp.Rational(1, 3))**n * sp.Heaviside(n)
H_z_inv = 1 / (1 - sp.Rational(1, 3) * z_inv)

print("\n1. Impulse response h[n] and Transfer Function H(z^-1):")
display(h_n)
display(H_z_inv)

# 2. Input Signal x[n] και ο Μετασχηματισμός Z X(z^-1)
x_n = (sp.Rational(1, 2))**n * sp.cos(sp.pi * n / sp.Rational(3, 1)) * sp.Heaviside(n)
X_z_inv = (1 - sp.Rational(1, 4) * z_inv) / (1 - sp.Rational(1, 2) * z_inv + sp.Rational(1, 4) * z_inv**2)

print("\n2. Input signal x[n] and its Z-transform X(z^-1):")
display(x_n)
display(X_z_inv)

# 3. Output Z-transform Y(z^-1) = H(z^-1) * X(z^-1)
Y_z_inv = sp.simplify(H_z_inv * X_z_inv)

print("\n3. Output Z-transform Y(z^-1) = H(z^-1) * X(z^-1):")
display(Y_z_inv)

# 4. Συμβολικός υπολογισμός μερικών κλασμάτων με επίλυση του συστήματος συντελεστών (A, B, C)
A = sp.Symbol('A')
B = sp.Symbol('B')
C = sp.Symbol('C')

pfe_general = A / (1 - sp.Rational(1, 3) * z_inv) + (B * z_inv + C) / (1 - sp.Rational(1, 2) * z_inv + sp.Rational(1, 4) * z_inv**2)
pfe_general_common = sp.simplify(pfe_general)

eq1 = sp.Eq(A + C, 1)
eq2 = sp.Eq(B - A/2 - C/3, -sp.Rational(1, 4))
eq3 = sp.Eq(A/4 - B/3, 0)

sol_abc = sp.solve((eq1, eq2, eq3), (A, B, C))
print("\nComputed Partial Fraction Coefficients (A, B, C):")
display(sol_abc)

pfe_y_inv = (sol_abc[A]) / (1 - sp.Rational(1, 3) * z_inv) + (sol_abc[B] * z_inv + sol_abc[C]) / (1 - sp.Rational(1, 2) * z_inv + sp.Rational(1, 4) * z_inv**2)

print("\n4. Partial Fraction Expansion of Y(z^-1):")
display(pfe_y_inv)

# 5. Ορθός αναλυτικός υπολογισμός εξόδου y[n] με καθαρή και έγκυρη σύνταξη
y_n_expr = ( sp.Rational(1, 7) * (sp.Rational(1, 3))**n + 
             sp.Rational(6, 7) * (sp.Rational(1, 2))**n * sp.cos(sp.pi * n / sp.Rational(3, 1)) + 
             (sp.Rational(3, 7) * sp.sqrt(3)) * (sp.Rational(1, 2))**n * sp.sin(sp.pi * n / sp.Rational(3, 1)) ) * sp.Heaviside(n)
y_n_simplified = sp.simplify(y_n_expr)

print("\n5. Final analytical output response y[n]:")
display(y_n_simplified)


# ==============================================================================
# 5. VISUALIZATION & INTERACTIVE PLOTS (Pole-Zero, Time Signals, Frequency Response)
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Interactive LTI System Analysis (Exact Symbolic Solution & Partial Fractions)</b><br>
* <b>Pole-Zero Map:</b> Displays system poles and zeros.<br>
* <b>Time Domain Signals:</b> Interactive plots for $h[n]$, $x[n]$, and output $y[n]$.<br>
* <b>Frequency Response:</b> Magnitude (dB) and Phase (degrees) across normalized frequency.<br>
* <b>Control:</b> Use the slider below to adjust the time range index <i>n</i>.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

def plot_system_analysis(n_max):
    with out:
        clear_output(wait=True)
        
        fig = plt.figure(figsize=(16, 14))
        gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1.1])
        plt.subplots_adjust(wspace=0.25, hspace=0.35)

        ax_pz = fig.add_subplot(gs[0, 0])
        ax_freq = fig.add_subplot(gs[0, 1])
        
        ax_h = fig.add_subplot(gs[1, 0])
        ax_x = fig.add_subplot(gs[1, 1])
        ax_y = fig.add_subplot(gs[2, :])

        # --- A. Pole-Zero Map for H(z^-1) ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-1.5, 1.5)
        ax_pz.set_ylim(-1.5, 1.5)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')

        ax_pz.scatter([1/3], [0], s=140, color='purple', marker='x', linewidths=3, label='Poles')
        ax_pz.scatter([0], [0], s=120, facecolors='none', edgecolors='blue', linewidths=2, marker='o', label='Zeros')

        ax_pz.set_title('System Pole-Zero Map', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)
        ax_pz.legend(loc='upper right', fontsize=8)

        # --- B. Frequency Response / Bode Plots ---
        omega = np.linspace(0, np.pi, 500)
        z_inv_val = np.exp(-1j * omega)
        
        H_omega = 1.0 / (1.0 - (1/3) * z_inv_val)
        
        mag_abs = np.abs(H_omega)
        mag_db = 20 * np.log10(np.maximum(mag_abs, 1e-12))
        phase_deg = np.angle(H_omega, deg=True)
        
        ax_freq.plot(omega / np.pi, mag_db, 'b-', linewidth=2, label='Magnitude (dB)')
        ax_freq.set_ylabel('Magnitude (dB)', color='b', fontsize=10)
        ax_freq.set_xlabel(r'Normalized Frequency ($\times \pi$ rad/sample)', fontsize=10)
        ax_freq.grid(True, linestyle=':', alpha=0.7)
        
        ax_phase = ax_freq.twinx()
        ax_phase.plot(omega / np.pi, phase_deg, 'r--', linewidth=2, label='Phase (deg)')
        ax_phase.set_ylabel('Phase (degrees)', color='r', fontsize=10)
        ax_freq.set_title('Frequency Response (Bode Plot)', fontsize=10, fontweight='bold')

        # --- C. Time Domain Signals (h[n], x[n], y[n]) ---
        n_vec = np.arange(0, n_max + 1)
        
        h_vals = np.array([float(h_n.subs(n, val).evalf()) for val in n_vec])
        x_vals = np.array([float(x_n.subs(n, val).evalf()) for val in n_vec])
        y_vals = np.array([float(y_n_simplified.subs(n, val).evalf()) for val in n_vec])

        # Impulse Response h[n]
        ax_h.stem(n_vec, h_vals, linefmt='g-', markerfmt='go', basefmt='k-')
        ax_h.set_title('Impulse Response: h[n]', fontsize=10, fontweight='bold')
        ax_h.set_xlabel('n', fontsize=9)
        ax_h.set_ylabel('h[n]', fontsize=9)
        ax_h.grid(True, linestyle=':', alpha=0.7)

        # Input Signal x[n]
        ax_x.stem(n_vec, x_vals, linefmt='b-', markerfmt='bo', basefmt='k-')
        ax_x.set_title('Input Signal: x[n]', fontsize=10, fontweight='bold')
        ax_x.set_xlabel('n', fontsize=9)
        ax_x.set_ylabel('x[n]', fontsize=9)
        ax_x.grid(True, linestyle=':', alpha=0.7)

        # Output Signal y[n]
        ax_y.stem(n_vec, y_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_y.set_title('Output Response: y[n] (Exact Symbolic Solution)', fontsize=10, fontweight='bold')
        ax_y.set_xlabel('Time index n', fontsize=9)
        ax_y.set_ylabel('y[n]', fontsize=9)
        ax_y.grid(True, linestyle=':', alpha=0.7)

        plt.show()

        print("-" * 115)
        print("ANALYSIS EXECUTED SUCCESSFULLY: Exact symbolic derivation matching reference solution.")
        print("-" * 115)

# Slider for time range n
n_slider = widgets.IntSlider(value=15, min=5, max=40, step=1, description='Max n:', style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'))

interactive_plot = widgets.interactive(plot_system_analysis, n_max=n_slider)
display(widgets.VBox([interactive_plot, out]))